In [59]:
import pandas as pd
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split

In [60]:
#load cleaned abo list
cleaned = pd.read_csv("cleaned_productdataset.csv.")
twentyk = pd.read_csv("abo_test_products_20000_final.csv")

#check how many rows and columns
print(cleaned.shape)
print(twentyk.shape)

#check column names
print(cleaned.columns)
print(twentyk.columns)

#check how many of each column is unique
print(cleaned.nunique())
print(twentyk.nunique())

#check how much of cleaned id's are the same as twentyk
print(len(set(cleaned["item_id"])& set(twentyk["item_id"])))

#check first 5 to see if id formats are the same
print(cleaned["item_id"].head(5).tolist())
print(twentyk["item_id"].head(5).tolist())

#twentyk has first id starting with 0
#check the rest of the id's starting values
print(twentyk["item_id"].str[0].value_counts())


(121554, 4)
(20000, 1)
Index(['item_id', 'description', 'main_image_id', 'image_path'], dtype='object')
Index(['item_id'], dtype='object')
item_id          121554
description      117309
main_image_id    108288
image_path       108288
dtype: int64
item_id    20000
dtype: int64
16598
['B000HCR89W', 'B000SMNJ6M', 'B000W48HL8', 'B000W4AEE6', 'B007XAKFUK']
['0615916821', 'B000050415', 'B00005041T', 'B0000B35D3', 'B0006Q5UT4']
item_id
B    19999
0        1
Name: count, dtype: int64


In [61]:
#remove the ids that are in twentyk before split 
# == False keeps the ones that are not in twentyk
kept = cleaned[cleaned["item_id"].isin(twentyk["item_id"]) == False]

#check how many we have left after removals
print(len(kept))

#transfer kept to csv file 
#index = False stops first column being numbered
kept.to_csv("kept_products.csv", index=False)
pd.read_csv("kept_products.csv").shape

#check no ids from twentyk are in kept 
print(len(kept))
print(len(set(kept["item_id"]) & set(twentyk["item_id"])))

104956
104956
0


In [62]:
kept = pd.read_csv("kept_products.csv")
print(pd.DataFrame(kept["main_image_id"]).nunique())

#check the number of unique main images 
#use main_image to prevent data leakage
#store unique images first using 80/20 split
unique_img= kept["main_image_id"].unique()
train_img, val_img = train_test_split(unique_img, test_size = 0.2, random_state=88)

#add remaning images 
train = kept[kept["main_image_id"].isin(train_img)]
val = kept[kept["main_image_id"].isin(val_img)]

#check numbers add up
print("train:", len(train))
print("val:", len(val))
print("sum:", len(train) + len(val))

#check none of the images or items are in both train and val sets
#should be 0
print("images in both splits:", len(set(train["main_image_id"]) & set (val["main_image_id"])))
print("items in both splits", len(set(train["item_id"]) & set(val["item_id"])))

train.to_csv("train_products.csv", index=False)
val.to_csv("val_products.csv", index=False)

main_image_id    94387
dtype: int64
train: 83523
val: 21433
sum: 104956
images in both splits: 0
items in both splits 0


In [63]:
#create positive pairs for training and validation
train_pos = train[["image_path", "description"]].copy()
train_pos["label"] = 1 
val_pos = val[["image_path", "description"]].copy()
val_pos["label"] = 1 

print(len(train_pos), len(val_pos))
print(train_pos.head())

83523 21433
                                      image_path  \
2  abo-images-small/images/small/01/0190eb0b.jpg   
4  abo-images-small/images/small/a1/a1e5d653.jpg   
5  abo-images-small/images/small/81/81a22dd4.jpg   
6  abo-images-small/images/small/92/9256b55e.jpg   
8  abo-images-small/images/small/7c/7ce58e4e.jpg   

                                         description  label  
2  Classic Accessories 70972 Veranda Patio Square...      1  
4  Waterdrop RO Reverse Osmosis Water Filtration ...      1  
5  [Amazon Brand] Eono Smartphone Waterproof Case...      1  
6  [Amazon Brand] Eono Smartphone Waterproof Case...      1  
8  Classic Accessories 70932 Veranda Rectangular/...      1  


In [64]:
import random
random.seed(88)

#turn descriptions into list and check the number
descr = train["description"].tolist()
n = len(descr)

#when j == i make j != i so descr[j] is never descr[i]
#so image path never has its own description
neg_descr = []
for i in range(n):
    j = i
    while j == i:
        j = random.randrange(n)
    neg_descr.append(descr[j])

#store image path and descr in train neg
train_neg = train[["image_path"]].copy()
train_neg["description"] = neg_descr
train_neg["label"] = 0

#check negative doesnt use its own description for image path
#should be 0
print(len(train_neg))
print("self pairs:", (train_neg["description"].values == train["description"].values).sum())

#do the same for validation set
descr = val["description"].tolist()
n = len(descr)

neg_descr = []
for i in range(n):
    j = i
    while j == i:
        j = random.randrange(n)
    neg_descr.append(descr[j])

val_neg = val[["image_path"]].copy()
val_neg["description"] = neg_descr
val_neg["label"] = 0

print(len(val_neg))
print("self pairs:", (val_neg["description"].values == val["description"].values).sum())

83523
self pairs: 0
21433
self pairs: 0


In [65]:
#stack the datadrames on top of each other 
train_pairs = pd.concat([train_pos, train_neg], ignore_index=True)
val_pairs = pd.concat([val_pos, val_neg], ignore_index=True)

#shuffles the data 
#resets number indices and drops the old ones
train_pairs = train_pairs.sample(frac=1, random_state = 88).reset_index(drop=True)
val_pairs = val_pairs.sample(frac=1, random_state = 88).reset_index(drop=True)

#prints positive and negative pairs 
#should be 50/50
print(len(train_pairs), train_pairs["label"].value_counts().to_dict())
print(len(val_pairs), val_pairs["label"].value_counts().to_dict())

167046 {0: 83523, 1: 83523}
42866 {0: 21433, 1: 21433}


In [66]:
#save the tables no row indices
train_pairs.to_csv("train_pairs.csv.gz", index=False)
val_pairs.to_csv("val_pairs.csv.gz", index=False)

print(pd.read_csv("train_pairs.csv.gz").shape)
print(pd.read_csv("val_pairs.csv.gz").shape)

#checks if any of the image paths are in both train and val sets
#should be 0
print("images in both splits:", len(set(train_pairs["image_path"]) & set(val_pairs["image_path"])))

(167046, 3)
(42866, 3)
images in both splits: 0
